# Notebook 04 — OFAC Cascading Match

This notebook demonstrates the four-stage cascading matching pipeline used to
determine whether a Comprehend-detected entity appears on the OFAC SDN list.

**What this covers:**
- Stage 1: Exact match (case-insensitive)
- Stage 2: Fuzzy matching with `rapidfuzz.token_sort_ratio`
- Stage 3: Semantic embedding with `sentence-transformers` + FAISS L1 index
- Stage 4: Composite scoring (fuzzy 0.4 + semantic 0.4 + substring 0.2)
- Routing: CONFIRMED / REVIEW / NO_MATCH

**Why a cascade:** Exact match is fast but misses typos.  Fuzzy match catches
near-matches but can be fooled by word order.  Semantic embedding captures
meaning similarity.  The composite scorer combines all three signals for a
robust final score.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer

from ner_ofac import config
from ner_ofac.analysis.faiss_index import build_index, search_index
from ner_ofac.analysis.ofac_matcher import match_entity, MatchResult

## 1. Setup: OFAC Name Lists and FAISS Index

We use a small mock OFAC list.  In production these come from `load_ofac()`.

In [ ]:
PERSON_NAMES = [
    "JOHN DOE",
    "JANE SMITH",
    "IVAN PETROV",
    "ALI HASSAN KHALID",
    "CARLOS MENDEZ GARCIA",
]

ORG_NAMES = [
    "ACME CORP",
    "SHADOW TRADING LLC",
    "DARK STAR VESSEL",
    "NORTHERN FINANCE GROUP",
]

print(f"Person names: {PERSON_NAMES}")
print(f"Org names:    {ORG_NAMES}")

In [ ]:
# Load embedding model and build FAISS indexes.
print(f"Loading model: {config.EMBEDDING_MODEL}")
model = SentenceTransformer(config.EMBEDDING_MODEL)

person_index, person_names = build_index(PERSON_NAMES, model)
org_index, org_names = build_index(ORG_NAMES, model)

print(f"Person index: {person_index.ntotal} vectors")
print(f"Org index:    {org_index.ntotal} vectors")

## 2. Stage 1: Exact Match

The first stage checks for a case-insensitive exact match.  This is the
highest confidence path and immediately routes to CONFIRMED.

In [ ]:
result = match_entity(
    entity_text="JOHN DOE",
    entity_type="SANCTIONED_PERSON",
    ofac_names=PERSON_NAMES,
    faiss_index=person_index,
    name_list=person_names,
    model=model,
)
print(f"Entity:         {result.entity_text}")
print(f"Matched:        {result.matched_name}")
print(f"Composite score:{result.composite_score:.3f}")
print(f"Stage:          {result.stage}")
print(f"Routing:        {result.routing}")

## 3. Stage 2: Fuzzy Matching

Fuzzy matching uses `rapidfuzz.fuzz.token_sort_ratio`.  This handles minor
typos, word order variations, and single-character edits.

In [ ]:
# Demonstrate fuzzy scores for near-matches.
test_queries = [
    "JOHN DO",          # 1 char deletion
    "DOE JOHN",         # word order swapped
    "JOHN DOES",        # 1 char addition
    "IVAN PETROV JR",   # extra token
    "WIDGET FACTORY",   # unrelated
]

print(f"{'Query':<30} {'Best OFAC Name':<30} {'Fuzzy Score':>12}")
print("-" * 75)
for query in test_queries:
    scores = [(name, fuzz.token_sort_ratio(query, name)) for name in PERSON_NAMES]
    best_name, best_score = max(scores, key=lambda x: x[1])
    flag = " ✓" if best_score >= config.FUZZY_THRESHOLD else ""
    print(f"{query:<30} {best_name:<30} {best_score:>10.0f}{flag}")

## 4. Stage 3: Semantic FAISS Search

The sentence-transformer encodes the query and FAISS performs an L1 nearest-
neighbour search.  Lower distance = more semantically similar.

In [ ]:
query = "SHADOW TRADING"
query_vec = model.encode([query], convert_to_numpy=True)[0].astype(np.float32)

results = search_index(org_index, org_names, query_vec, k=4)
print(f"Top-4 semantic matches for '{query}':")
print(f"{'OFAC Name':<30} {'L1 Distance':>12}")
print("-" * 45)
for name, dist in results:
    print(f"{name:<30} {dist:>12.4f}")

## 5. Stage 4: Composite Scoring and Routing

The composite score combines:
- `FUZZY_WEIGHT (0.4)` × normalised fuzzy score
- `SEMANTIC_WEIGHT (0.4)` × normalised semantic similarity
- `0.2` × exact substring bonus (1.0 if entity appears in OFAC name, else 0.0)

Routing thresholds:
- `≥ 0.90` → **CONFIRMED**
- `0.65 – 0.89` → **REVIEW**
- `< 0.65` → **NO_MATCH**

In [ ]:
test_entities = [
    ("JOHN DOE",        "SANCTIONED_PERSON", PERSON_NAMES, person_index, person_names),
    ("JOHN DO",         "SANCTIONED_PERSON", PERSON_NAMES, person_index, person_names),
    ("SHADOW TRADING",  "SANCTIONED_ORG",    ORG_NAMES,    org_index,    org_names),
    ("WIDGET FACTORY",  "SANCTIONED_ORG",    ORG_NAMES,    org_index,    org_names),
    ("IVAN PETROV",     "SANCTIONED_PERSON", PERSON_NAMES, person_index, person_names),
]

print(f"{'Entity':<25} {'Matched':<25} {'Score':>7} {'Stage':<20} {'Routing'}")
print("-" * 95)
for entity_text, entity_type, names, idx, nl in test_entities:
    r = match_entity(entity_text, entity_type, names, idx, nl, model)
    print(f"{entity_text:<25} {str(r.matched_name):<25} {r.composite_score:>7.3f} {r.stage:<20} {r.routing}")

## 6. Category Silo Demonstration

SANCTIONED_PERSON entities only search person names, and SANCTIONED_ORG
entities only search org names.  This prevents cross-category false positives.

In [ ]:
# Try matching a person name against the org name list (wrong silo).
result_wrong_silo = match_entity(
    entity_text="JOHN DOE",
    entity_type="SANCTIONED_ORG",   # wrong silo
    ofac_names=ORG_NAMES,
    faiss_index=org_index,
    name_list=org_names,
    model=model,
)
print("Person name matched against ORG list:")
print(f"  Score={result_wrong_silo.composite_score:.3f}, Routing={result_wrong_silo.routing}")

# Now match against the correct person list.
result_correct_silo = match_entity(
    entity_text="JOHN DOE",
    entity_type="SANCTIONED_PERSON",
    ofac_names=PERSON_NAMES,
    faiss_index=person_index,
    name_list=person_names,
    model=model,
)
print("\nSame name matched against PERSON list (correct silo):")
print(f"  Score={result_correct_silo.composite_score:.3f}, Routing={result_correct_silo.routing}")

## Summary

- Stage 1 (exact): Fastest path, highest confidence → CONFIRMED at composite=1.0
- Stage 2 (fuzzy): Catches typos and word order variations
- Stage 3 (FAISS): Captures semantic similarity via embedding space
- Stage 4 (composite): Combines all signals with configured weights
- Routing thresholds drive downstream action: auto-accept, human review, or discard